# Running Tafsut on the GIFT-Eval benchmark

This notebook reproduces GIFT-Eval results for the univariate Tafsut foundation model
`Tafsut-FM/tafsut-univariate-base` using a GluonTS-style predictor interface.

Make sure the GIFT-Eval benchmark is installed and its dataset environment variable is
configured before running this notebook. The notebook follows the same dataset/term
evaluation structure as the Chronos-2 submission notebook.

Tafsut is a **univariate** model. For GIFT-Eval datasets whose source target is multivariate,
the dataset is converted to univariate series with `to_univariate=True`.


Install Tafsut and the GIFT-Eval dependencies if needed:

```bash
pip install tafsut
```

Run this notebook from the GIFT-Eval repository (or adjust `GIFT_EVAL_SRC` and
`DATASET_PROPERTIES_PATH` below).


In [1]:
import json
import logging
import os
import sys
import warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch

warnings.simplefilter("ignore")

GIFT_EVAL_SRC = os.environ.get("GIFT_EVAL_SRC", "../src")
DATASET_PROPERTIES_PATH = os.environ.get(
    "GIFT_EVAL_DATASET_PROPERTIES",
    "dataset_properties.json",
)

# gift_eval.data.Dataset reads the dataset root from the GIFT_EVAL environment
# variable. Respect an existing value; otherwise use the standard sibling
# directory when this notebook is run from the gift-eval repository.
GIFT_EVAL_DATA = os.environ.get("GIFT_EVAL", "dataset")
os.environ["GIFT_EVAL"] = GIFT_EVAL_DATA

dataset_root = Path(GIFT_EVAL_DATA).expanduser()
if not dataset_root.exists():
    raise RuntimeError(
        "GIFT-Eval dataset directory was not found. "
        f"GIFT_EVAL={GIFT_EVAL_DATA!r}. "
        "Set GIFT_EVAL to the directory containing the downloaded GIFT-Eval datasets "
        "before running the evaluation cells."
    )

print("GIFT_EVAL dataset root:", dataset_root.resolve())

sys.path.insert(0, GIFT_EVAL_SRC)

from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import Forecast, evaluate_forecasts
from gluonts.model.forecast import QuantileForecast
from gluonts.time_feature import get_seasonality

from gift_eval.data import Dataset
from tafsut import TafsutModel, forecast

model_name = "Tafsut-FM/tafsut-univariate-base"

pretty_model_name = "tafsut"
if torch.cuda.is_available():
    DEVICE = os.environ.get("TAFSUT_DEVICE", "cuda:0")
else:
    DEVICE = "cpu"

INFERENCE_BATCH_SIZE = int(os.environ.get("TAFSUT_BATCH_SIZE", "32"))
EVALUATION_BATCH_SIZE = int(os.environ.get("GIFT_EVAL_BATCH_SIZE", "1024"))

output_dir = "../results/tafsut/all_results.csv"
os.makedirs(os.path.dirname(output_dir), exist_ok=True)

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

SHORT_DATASETS = (
    "temperature_rain_with_missing m4_yearly m4_quarterly m4_monthly "
    "m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D "
    "electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths "
    "us_births/D us_births/M us_births/W saugeenday/D saugeenday/M "
    "saugeenday/W kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D "
    "car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W "
    "LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H "
    "M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H "
    "ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D "
    "bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T "
    "bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T "
    "bizitobs_l2c/H"
)

MED_LONG_DATASETS = (
    "electricity/15T electricity/H solar/10T solar/H "
    "kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H "
    "SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H "
    "jena_weather/10T jena_weather/H bitbrains_fast_storage/5T "
    "bitbrains_rnd/5T bizitobs_application bizitobs_service "
    "bizitobs_l2c/5T bizitobs_l2c/H"
)

all_datasets = list(
    dict.fromkeys(SHORT_DATASETS.split() + MED_LONG_DATASETS.split())
)

with open(DATASET_PROPERTIES_PATH, "r") as f:
    dataset_properties_map = json.load(f)

# Load a single Tafsut model.
print(f"Loading Tafsut on {DEVICE} ...")
model = TafsutModel.from_pretrained(
    model_name,
    device=DEVICE,
)
model.eval()

METRIC_QUANTILE_LEVELS = [float(q) for q in model.cfg.quantiles]

print("Device:", DEVICE)
print("Inference batch size:", INFERENCE_BATCH_SIZE)
print("Metric evaluation batch size:", EVALUATION_BATCH_SIZE)
print("Tafsut quantiles:", METRIC_QUANTILE_LEVELS)


metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=METRIC_QUANTILE_LEVELS
    ),
]


GIFT_EVAL dataset root: /home/t00876569/SFM/gift/dataset
Loading Tafsut on cuda:0 ...
Device: cuda:0
Inference batch size: 512
Metric evaluation batch size: 16384
Tafsut quantiles: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


In [2]:
logger = logging.getLogger("Tafsut Predictor")
logger.setLevel(logging.INFO)


class TafsutPredictor:
    """
    Simple batched Tafsut predictor for GIFT-Eval.

    GIFT series can have different lengths, so each batch is left-padded with NaN
    to form a rectangular (B, T) array. Tafsut handles validity masking and context
    truncation internally.
    """

    def __init__(
        self,
        model: TafsutModel,
        prediction_length: int,
        batch_size: int = 128,
    ):
        self.model = model
        self.prediction_length = int(prediction_length)
        self.batch_size = int(batch_size)
        self.quantile_levels = [float(q) for q in model.cfg.quantiles]

    def _pad_contexts(self, entries):
        max_context = int(self.model.cfg.context_length)

        # Pre-truncate for efficiency. Tafsut applies the same right-side
        # truncation internally, so this does not change model semantics;
        # it only avoids padding/transferring history the model will discard.
        contexts = [
            np.asarray(entry["target"], dtype=np.float32).reshape(-1)[-max_context:]
            for entry in entries
        ]

        max_len = max(len(x) for x in contexts)

        return np.stack([
            np.pad(
                x,
                (max_len - len(x), 0),
                mode="constant",
                constant_values=np.nan,
            )
            for x in contexts
        ])

    def _forecast_batch(self, entries):
        contexts = self._pad_contexts(entries)

        output = forecast(
            self.model,
            contexts,
            horizon=self.prediction_length,
        )

        if torch.is_tensor(output):
            output = output.numpy()
        else:
            output = np.asarray(output)

        expected = (
            len(entries),
            self.prediction_length,
            len(self.quantile_levels),
        )
        if output.shape != expected:
            raise ValueError(
                f"Unexpected Tafsut forecast shape {output.shape}; "
                f"expected {expected} = (batch, horizon, quantiles)."
            )

        return output

    def predict(self, test_data_input) -> List[Forecast]:
        entries = list(test_data_input)
        forecasts: List[Forecast] = []

        for start in range(0, len(entries), self.batch_size):
            batch = entries[start:start + self.batch_size]

            try:
                predictions = self._forecast_batch(batch)
            except torch.cuda.OutOfMemoryError as exc:
                raise RuntimeError(
                    f"CUDA OOM. Reduce TAFSUT_BATCH_SIZE "
                    f"(current value: {self.batch_size})."
                ) from exc

            for prediction, entry in zip(predictions, batch):
                forecasts.append(
                    QuantileForecast(
                        forecast_arrays=prediction.T,
                        forecast_keys=[str(q) for q in self.quantile_levels],
                        start_date=entry["start"] + len(entry["target"]),
                    )
                )

        return forecasts


## Batched inference

This notebook uses a single Tafsut model on one device.

GIFT series are processed sequentially in batches. Each history is first
pre-truncated to the model's configured context length, then variable-length
histories are left-padded with `NaN` so they can be represented as a rectangular
`(B, T)` array. Tafsut still handles finite-value masking and performs the same
context truncation internally; pre-truncation here only avoids moving data that
the model would immediately discard.

Configuration:

- `TAFSUT_DEVICE=cuda:0` selects the GPU. If CUDA is unavailable, CPU is used.
- `TAFSUT_BATCH_SIZE=128` controls the model inference batch size.
- `GIFT_EVAL_BATCH_SIZE=1024` controls only GluonTS metric computation after
  forecasts have been generated.

There are no worker threads, queues, multiprocessing processes, or model replicas.


## Evaluation

The evaluator follows the GIFT-Eval dataset naming convention:

```python
f"{dataset_name}/{freq}/{term}"
```

Tafsut is evaluated as a univariate model. If a GIFT-Eval source dataset is
multivariate, `Dataset(..., to_univariate=True)` converts it into separate
univariate series before forecasting.

The complete GIFT input stream is forecast in its native order. Batches can therefore
span rolling-window boundaries, avoiding small per-window inference calls while preserving
the ordering expected by `dataset.test_data` during metric computation.


In [3]:
class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger("gluonts.model.forecast")
gts_logger.addFilter(
    WarningFilter("The mean prediction is not stored in the forecast data")
)


In [4]:
def evaluate_on_dataset(
    model: TafsutModel,
    ds_name: str,
    ds_term: str,
    batch_size: int,
):
    is_multivariate_source = (
        Dataset(
            name=ds_name,
            term=ds_term,
            to_univariate=False,
        ).target_dim
        > 1
    )

    dataset = Dataset(
        name=ds_name,
        term=ds_term,
        to_univariate=is_multivariate_source,
    )

    print(
        f"Dataset size: {len(dataset.test_data)} | "
        f"prediction_length={dataset.prediction_length}"
    )

    predictor = TafsutPredictor(
        model=model,
        prediction_length=dataset.prediction_length,
        batch_size=batch_size,
    )

    # Forecast the complete GIFT input stream in its native order.
    # This is equivalent to the previous per-window split/re-interleave logic,
    # but allows batches to span rolling-window boundaries and keeps the GPU
    # fed with larger, more continuous batches.
    forecasts = predictor.predict(dataset.test_data.input)

    season_length = get_seasonality(dataset.freq)
    return (
        evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=EVALUATION_BATCH_SIZE,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )
        .reset_index(drop=True)
        .to_dict(orient="records")
    )


In [5]:
all_results = []

for ds_num, ds_name in enumerate(all_datasets):
    print(
        f"Processing dataset: {ds_name} "
        f"({ds_num + 1} of {len(all_datasets)})"
    )

    for term in ("short", "medium", "long"):
        if term in ("medium", "long") and ds_name not in MED_LONG_DATASETS.split():
            continue

        if "/" in ds_name:
            raw_key, ds_freq = ds_name.split("/", 1)
            ds_key = pretty_names.get(raw_key.lower(), raw_key.lower())
        else:
            raw_key = ds_name
            ds_key = pretty_names.get(raw_key.lower(), raw_key.lower())
            ds_freq = dataset_properties_map[ds_key]["frequency"]

        ds_config = f"{ds_key}/{ds_freq}/{term}"
        print(f"Generating forecasts for {ds_config}")

        all_results.append(
            (
                evaluate_on_dataset(
                    model=model,
                    ds_name=ds_name,
                    ds_term=term,
                    batch_size=INFERENCE_BATCH_SIZE,
                ),
                ds_config,
                dataset_properties_map[ds_key]["domain"],
                dataset_properties_map[ds_key]["num_variates"],
            )
        )


result_df_rows = []

for result_metrics, ds_config, domain, num_variates in all_results:
    result_metrics = {
        f"eval_metrics/{k}": v
        for k, v in result_metrics[0].items()
    }

    result_df_rows.append(
        {
            "dataset": ds_config,
            "model": pretty_model_name,
            **result_metrics,
            "domain": domain,
            "num_variates": num_variates,
        }
    )

results_df = pd.DataFrame(result_df_rows).sort_values(by="dataset")
results_df.to_csv(output_dir, index=False)

print(f"Results have been written to {output_dir}.")
results_df


Processing dataset: temperature_rain_with_missing (1 of 55)
Generating forecasts for temperature_rain/D/short
Dataset size: 96216 | prediction_length=30


96216it [01:07, 1423.42it/s]


Processing dataset: m4_yearly (2 of 55)
Generating forecasts for m4_yearly/A/short
Dataset size: 22974 | prediction_length=6


22974it [00:17, 1306.73it/s]


Processing dataset: m4_quarterly (3 of 55)
Generating forecasts for m4_quarterly/Q/short
Dataset size: 24000 | prediction_length=8


...

7it [00:00, 688.57it/s]


Generating forecasts for bizitobs_l2c/H/long
Dataset size: 7 | prediction_length=720


7it [00:00, 697.49it/s]

Results have been written to ../results/tafsut-univariate-base/all_results.csv.


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
79,bitbrains_fast_storage/5T/long,Tafsut-univariate-base,5.975722e+06,5.975722e+06,412.539461,0.887021,6.869519,0.769049,16.252108,2444.528938,6.460159,1.090218,0.781447,Web/CloudOps,2
78,bitbrains_fast_storage/5T/medium,Tafsut-univariate-base,4.679275e+06,4.679275e+06,316.777693,0.968898,6.430133,0.743791,21.308416,2163.163129,6.572286,0.962458,0.718490,Web/CloudOps,2
77,bitbrains_fast_storage/5T/short,Tafsut-univariate-base,2.060366e+06,2.060366e+06,163.994882,0.683290,2.498665,0.678000,13.928155,1435.397388,4.506793,0.514903,0.410832,Web/CloudOps,2
80,bitbrains_fast_storage/H/short,Tafsut-univariate-base,2.572213e+06,2.572213e+06,252.555313,0.947577,2.227298,0.525274,20.843571,1603.812164,4.571386,0.719865,0.583826,Web/CloudOps,2
83,bitbrains_rnd/5T/long,Tafsut-univariate-base,3.239946e+06,3.239946e+06,214.232963,3.303981,3.786258,0.661358,120.990723,1799.984965,6.895335,0.820678,0.702126,Web/CloudOps,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49,sz_taxi/H/short,Tafsut-univariate-base,7.166241e+00,7.166241e+00,1.841270,0.560389,1.114248,0.294755,3.982041,2.676984,0.249378,0.171526,0.135373,Transport,1
0,temperature_rain/D/short,Tafsut-univariate-base,1.855438e+02,1.855438e+02,5.829005,1.346218,17.049233,1.531280,20.264271,13.621446,1.603604,0.686228,0.554925,Nature,1
25,us_births/D/short,Tafsut-univariate-base,1.577508e+05,1.577508e+05,258.539036,0.380735,0.024906,0.024615,3.083915,397.178516,0.037232,0.024236,0.019695,Healthcare,1
26,us_births/M/short,Tafsut-univariate-base,4.215827e+07,4.215827e+07,5255.231771,0.593495,0.016308,0.016460,4.141884,6492.939961,0.020167,0.016323,0.013286,Healthcare,1
